# The Lookbook — free on-device Pro pack conversion (Google Colab)

**What this does:** converts the commercially-licensed SD1.5 + ControlNet-Depth + IP-Adapter
weights into the ONNX FP16 pack the Android app runs **on-device** — so photoreal generation is
**free, offline, and $0 per image** once installed. This is a **one-time** step.

**Cost:** free. Runtime → **Change runtime type → T4 GPU** (Colab's free tier is enough).

**Time:** ~30–45 min (download ~6 GB + export).

---
### Honest note before you start
Steps 1–5 (text encoder, VAE, ControlNet, depth, IP image encoder) export cleanly. The **fused UNet**
(step 6) is the hard part — diffusers' UNet + ControlNet residuals + IP-Adapter tokens is export-sensitive
and often needs **one iteration** to line the ONNX input names up with the Android runtime. If step 6 errors,
**copy the full error back to the chat** and it'll get fixed. Everything else will still have produced valid files.

## 0. Check the GPU

In [ ]:
import torch
assert torch.cuda.is_available(), (
    'No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again.'
)
print('GPU:', torch.cuda.get_device_name(0))

## 1. Install dependencies

In [ ]:
!pip -q install 'diffusers==0.31.0' 'transformers==4.44.2' accelerate safetensors \
    onnx 'onnxruntime-gpu' 'huggingface_hub[cli]' 2>/dev/null
print('deps installed')

## 2. Download the source weights (public, no token needed)
Realistic Vision V5.1 (OpenRAIL-M) · ControlNet-Depth (OpenRAIL) · IP-Adapter Plus + image encoder (Apache-2.0).
All three are commercially shippable — see `ml/MODEL_LICENSES.md`.

In [ ]:
from huggingface_hub import hf_hub_download
import os, pathlib
SRC = pathlib.Path('pro_src'); SRC.mkdir(exist_ok=True)

base = hf_hub_download('SG161222/Realistic_Vision_V5.1_noVAE',
    'Realistic_Vision_V5.1_fp16-no-ema.safetensors', local_dir=SRC/'realistic_vision')
controlnet = hf_hub_download('comfyanonymous/ControlNet-v1-1_fp16_safetensors',
    'control_v11f1p_sd15_depth_fp16.safetensors', local_dir=SRC/'controlnet_depth')
for f in ['models/ip-adapter-plus_sd15.safetensors',
          'models/image_encoder/model.safetensors',
          'models/image_encoder/config.json']:
    hf_hub_download('h94/IP-Adapter', f, local_dir=SRC/'ip_adapter')
print('sources ready in', SRC)

## 3. Load the pipeline (base + ControlNet + IP-Adapter)

In [ ]:
import torch, pathlib
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel

SRC = pathlib.Path('pro_src')
OUT = pathlib.Path('exports/pro-v1'); OUT.mkdir(parents=True, exist_ok=True)
IMG, LAT = 512, 64
dtype, device = torch.float16, 'cuda'

pipe = StableDiffusionControlNetPipeline.from_single_file(
    str(SRC/'realistic_vision'/'Realistic_Vision_V5.1_fp16-no-ema.safetensors'),
    controlnet=ControlNetModel.from_single_file(
        str(SRC/'controlnet_depth'/'control_v11f1p_sd15_depth_fp16.safetensors'), torch_dtype=dtype),
    torch_dtype=dtype, safety_checker=None).to(device)
pipe.load_ip_adapter(str(SRC/'ip_adapter'/'models'), subfolder='',
    weight_name='ip-adapter-plus_sd15.safetensors',
    image_encoder_folder=str(SRC/'ip_adapter'/'models'/'image_encoder'))
pipe.set_ip_adapter_scale(0.7)
print('pipeline loaded')

## 4. Export the clean components (text encoder, VAE, ControlNet, IP image encoder)

In [ ]:
import torch
OP = 17

# --- text encoder ---
torch.onnx.export(pipe.text_encoder,
    (torch.ones(1, 77, dtype=torch.int64, device=device),),
    str(OUT/'text_encoder.onnx'), input_names=['input_ids'],
    output_names=['embeddings'], opset_version=OP)

# --- VAE enc/dec ---
class Enc(torch.nn.Module):
    def __init__(s, vae): super().__init__(); s.vae = vae
    def forward(s, x): return s.vae.encode(x).latent_dist.mode() * s.vae.config.scaling_factor
class Dec(torch.nn.Module):
    def __init__(s, vae): super().__init__(); s.vae = vae
    def forward(s, z): return s.vae.decode(z / s.vae.config.scaling_factor).sample
torch.onnx.export(Enc(pipe.vae), (torch.zeros(1,3,IMG,IMG,dtype=dtype,device=device),),
    str(OUT/'vae_encoder.onnx'), input_names=['image'], output_names=['latent'], opset_version=OP)
torch.onnx.export(Dec(pipe.vae), (torch.zeros(1,4,LAT,LAT,dtype=dtype,device=device),),
    str(OUT/'vae_decoder.onnx'), input_names=['latent'], output_names=['image'], opset_version=OP)

# --- ControlNet-Depth ---
ctrl_out = [f'down_{i}' for i in range(12)] + ['mid']
torch.onnx.export(pipe.controlnet,
    (torch.zeros(1,4,LAT,LAT,dtype=dtype,device=device),
     torch.tensor([1],dtype=torch.int64,device=device),
     torch.zeros(1,77,768,dtype=dtype,device=device),
     torch.zeros(1,3,IMG,IMG,dtype=dtype,device=device)),
    str(OUT/'controlnet.onnx'),
    input_names=['sample','timestep','encoder_hidden_states','controlnet_cond'],
    output_names=ctrl_out, opset_version=OP)

# --- IP-Adapter image encoder (CLIP-H) ---
torch.onnx.export(pipe.image_encoder,
    (torch.zeros(1,3,224,224,dtype=dtype,device=device),),
    str(OUT/'ip_image_encoder.onnx'), input_names=['image'],
    output_names=['image_embeds'], opset_version=OP)
print('clean components exported:', sorted(p.name for p in OUT.glob('*.onnx')))

## 5. Export the depth estimator (Stage A structural map)

In [ ]:
import torch
from transformers import AutoModelForDepthEstimation
SIZE = 518
dm = AutoModelForDepthEstimation.from_pretrained('depth-anything/Depth-Anything-V2-Small-hf').eval().to(device).half()
class Depth(torch.nn.Module):
    def __init__(s, m): super().__init__(); s.m = m
    def forward(s, x):
        d = s.m(pixel_values=x).predicted_depth.unsqueeze(1)
        lo = d.amin((2,3),keepdim=True); hi = d.amax((2,3),keepdim=True)
        return (d - lo) / (hi - lo + 1e-6)
torch.onnx.export(Depth(dm), (torch.zeros(1,3,SIZE,SIZE,dtype=dtype,device=device),),
    str(OUT/'depth.onnx'), input_names=['image'], output_names=['depth'], opset_version=OP,
    dynamic_axes={'image':{0:'b'},'depth':{0:'b'}})
print('depth.onnx exported')

## 6. Export the IP-Adapter projection + the fused UNet  ⚠️ the tricky step
The UNet takes the ControlNet residuals and the IP-Adapter image tokens. If this cell errors,
**paste the full traceback back into the chat** — the diffusers export API shifts between versions
and this is where a one-line fix usually lands. The files from steps 4–5 are already valid.

In [ ]:
import torch

# 6a. IP-Adapter projection (image_embeds -> cross-attn tokens). Lives on the UNet
#     after load_ip_adapter; export it alone so the app doesn't re-run CLIP per step.
proj = getattr(pipe.unet, 'encoder_hid_proj', None)
assert proj is not None, 'IP-Adapter projection not found (load_ip_adapter failed?)'
# IP-Adapter Plus resampler consumes CLIP-H penultimate hidden states [1,257,1280].
torch.onnx.export(proj, ([torch.zeros(1,257,1280,dtype=dtype,device=device)],),
    str(OUT/'ip_proj.onnx'), input_names=['image_embeds'], output_names=['ip_tokens'], opset_version=OP)
print('ip_proj.onnx exported')

# 6b. Fused UNet. image_embeds go through added_cond_kwargs (diffusers' IP-Adapter path),
#     ControlNet residuals through the additional-residual kwargs. We probe the real
#     residual shapes from ControlNet so the ONNX inputs are correctly shaped.
with torch.no_grad():
    down_res, mid_res = pipe.controlnet(
        torch.zeros(1,4,LAT,LAT,dtype=dtype,device=device),
        torch.tensor([1],dtype=torch.int64,device=device),
        encoder_hidden_states=torch.zeros(1,77,768,dtype=dtype,device=device),
        controlnet_cond=torch.zeros(1,3,IMG,IMG,dtype=dtype,device=device),
        return_dict=False)
down_names = [f'down_{i}' for i in range(len(down_res))]
print('residual shapes:', [tuple(r.shape) for r in down_res], '| mid', tuple(mid_res.shape))

class UNetFused(torch.nn.Module):
    def __init__(s, unet): super().__init__(); s.unet = unet
    def forward(s, sample, timestep, encoder_hidden_states, image_embeds, *residuals):
        down = list(residuals[:-1]); mid = residuals[-1]
        return s.unet(sample, timestep, encoder_hidden_states,
            down_block_additional_residuals=down,
            mid_block_additional_residual=mid,
            added_cond_kwargs={'image_embeds': [image_embeds]},
            return_dict=False)[0]

ip_tokens = torch.zeros(1, 16, 768, dtype=dtype, device=device)  # IP-Adapter Plus -> 16 tokens
args = (torch.zeros(1,4,LAT,LAT,dtype=dtype,device=device),
        torch.tensor([1],dtype=torch.int64,device=device),
        torch.zeros(1,77,768,dtype=dtype,device=device),
        ip_tokens, *down_res, mid_res)
torch.onnx.export(UNetFused(pipe.unet), args, str(OUT/'unet.onnx'),
    input_names=['sample','timestep','encoder_hidden_states','image_embeds', *down_names, 'mid'],
    output_names=['noise_pred'], opset_version=OP)
print('unet.onnx exported ✅')

## 7. Write the pack config + manifest

In [ ]:
import json, hashlib, pathlib
OUT = pathlib.Path('exports/pro-v1')
HF_USER = 'Iamzakirzr'   # <- your HF username (dataset owner)
HF_DATASET = 'vestra-packs'
BASE_URL = f'https://huggingface.co/datasets/{HF_USER}/{HF_DATASET}/resolve/main'

(OUT/'config.json').write_text(json.dumps({
    'latentWidth':64,'latentHeight':64,'imageWidth':512,'imageHeight':512,'resolution':512,
    'inferenceSteps':22,'guidanceScale':7.0,'vaeScale':0.18215,'concatAxis':'width',
    'unet':'unet.onnx','textEncoder':'text_encoder.onnx','vaeEncoder':'vae_encoder.onnx',
    'vaeDecoder':'vae_decoder.onnx','controlNet':'controlnet.onnx','depthModel':'depth.onnx',
    'ipAdapter':'ip_proj.onnx','imageEncoder':'ip_image_encoder.onnx'}, indent=2))
(OUT/'pack.json').write_text(json.dumps({
    'version':1,'tier':'PRO','kind':'ENGINE',
    'displayName':'Pro engine (SD1.5 + ControlNet-Depth + IP-Adapter)',
    'description':'Photorealistic on-device try-on for flagship NPUs. Commercially licensed weights.',
    'minSpec':{'minRamMb':7168,'requiresNpu':True,'minSdk':31}}, indent=2))

def sha256(p):
    h = hashlib.sha256(); h.update(p.read_bytes()); return h.hexdigest()
files = [{'path':p.relative_to(OUT).as_posix(),'url':f'{BASE_URL}/pro-v1/{p.relative_to(OUT).as_posix()}',
          'sha256':sha256(p),'bytes':p.stat().st_size}
         for p in sorted(OUT.rglob('*')) if p.is_file() and p.name not in ('pack.json','manifest.json')]
manifest = {'schemaVersion':1,'packs':[{'id':'pro-v1','version':1,'tier':'PRO',
    'displayName':'Pro engine (SD1.5 + ControlNet-Depth + IP-Adapter)',
    'description':'Photorealistic on-device try-on for flagship NPUs.',
    'totalBytes':sum(f['bytes'] for f in files),'files':files,
    'minSpec':{'minRamMb':7168,'requiresNpu':True,'minSdk':31}}]}
pathlib.Path('exports/manifest.json').write_text(json.dumps(manifest, indent=2))
print(f"pack total: {manifest['packs'][0]['totalBytes']/1e6:.0f} MB across {len(files)} files")
for f in files: print(' ', f['path'], f"{f['bytes']/1e6:.1f} MB")

## 8. Upload to your Hugging Face dataset
Paste a **write** token from https://huggingface.co/settings/tokens when prompted.
This pushes `pro-v1/` (the ONNX pack) and `manifest.json` to `Iamzakirzr/vestra-packs`.

In [ ]:
from huggingface_hub import HfApi, login
login()  # paste WRITE token
api = HfApi()
api.create_repo(f'{HF_USER}/{HF_DATASET}', repo_type='dataset', exist_ok=True)
api.upload_folder(folder_path='exports/pro-v1', path_in_repo='pro-v1',
    repo_id=f'{HF_USER}/{HF_DATASET}', repo_type='dataset')
api.upload_file(path_or_fileobj='exports/manifest.json', path_in_repo='manifest.json',
    repo_id=f'{HF_USER}/{HF_DATASET}', repo_type='dataset')
print('uploaded ✅  manifest:', f'{BASE_URL}/manifest.json')

## 9. Wire the app
In `composeApp/src/main/kotlin/com/zakir/vestra/VestraApp.kt` set:
```kotlin
const val PACKS_MANIFEST_URL =
    "https://huggingface.co/datasets/Iamzakirzr/vestra-packs/resolve/main/manifest.json"
```
Rebuild. On a flagship phone the app offers the **Pro pack** in Settings → Model packs; install it once,
then every generation is **on-device, offline, and free**. Report back here if any export cell errored and
it'll get patched.

> Note: the app's on-device Pro runtime (`SdControlNetPipeline.kt`) expects the UNet inputs named exactly
> `sample, timestep, encoder_hidden_states, image_embeds, down_0..down_11, mid`. If step 6 printed different
> residual counts/shapes, share that line — it's a small runtime alignment, not a rebuild of the pipeline.